In [1]:
from passwords import *
from pprint import pprint
import pandas as pd
import pyodbc
import os
import boto3
import datetime as dt

In [2]:
print(f'Latest run date: {dt.datetime.today()}')

Latest run date: 2024-03-25 09:07:15.772963


### Functions

In [3]:
# upload to s3
def upload_to_s3(aws_access_key_id, aws_secret_access_key, str_local_path, str_bucket_key, str_bucket_name):
    # init client
    cls_client = boto3.client(
        's3',
        aws_access_key_id=aws_access_key_id,
        aws_secret_access_key=aws_secret_access_key,
        aws_session_token=None,
    )
    # upload
    cls_client.upload_file(
        str_local_path, 
        str_bucket_name, 
        str_bucket_key,
    )

### Constants

In [4]:
# project
str_project = os.getcwd().split('\\')[4].replace('_','-')
print(f'Project: {str_project}')
# task
str_task = os.getcwd().split('\\')[5]
print(f'Task: {str_task}')
# sub task
str_subtask = os.getcwd().split('\\')[6]
print(f'Subtask: {str_subtask}')
# output
str_dirname_output = './output'

Project: 20231010-gen-xii
Task: 12_dark_scoring
Subtask: 03_analysis


### Output directory

In [5]:
try:
    os.mkdir(str_dirname_output)
except FileExistsError:
    pass

### Read query

In [6]:
str_filepath = './sql/query.sql'
str_query = open(str_filepath, 'r').read()
pprint(str_query)

('with tbl1 as\n'
 '(\n'
 'select *,\n'
 'Row_number() OVER(partition BY intAccountKey, strScoreCardVersion ORDER BY '
 'DimScoreCard.dtmStampCreation desc) RowNum\n'
 'from riskdb.dbo.vwReportOpenBKApplications left outer join '
 'edw.pfsedw.dbo.DimScoreCard\n'
 'on vwReportOpenBKApplications.bigaccountid = DimScoreCard.intAccountKey\n'
 "where dtmStampCreation >= '2024-03-14'\n"
 "and strScoreCardVersion like 'genxii_v2'\n"
 ')\n'
 'select*\n'
 'from tbl1\n'
 'where RowNum = 1')


### Write into df

In [7]:
%%time

# create connection
conn = pyodbc.connect(
    'Driver={SQL Server};'
    'Server=electra;'
    'Database=pfsdb;'
    'Trusted_Connection=yes;'
)
df = pd.read_sql_query(
    str_query, 
    con=conn,
)
# close
conn.close()

# show
df

Wall time: 9.53 s


,bigaccountid,intType,intScoreCardKey,intAccountKey,strTier,fltDebtorScore,fltCoDebtorScore,strScoreCardVersion,dtmStampCreation,intPayLoadTime,...,fltCoDebtor_Score_lgd,fltCoDebtor_Score_pd,fltCoDebtor_Score_ad,strS3StorageURL,dtmCreateDate,dtmUpdateDate,strCreateUser,strUpdateUser,intAuditKey,RowNum
0,7034142,13,65f88bd74800b0c9b782b364,7034142,A1,0.013755,0.000000,genxii_v2,2024-03-18 18:45:43.493,4614,...,0.000000,0.000000,0.000000,N/A,2024-03-18 22:50:54.973,2024-03-24 22:56:15.210,GOPFS\gmsaprodsql$,GOPFS\gmsaprodsql$,3207,1
1,7086394,13,65f88bef43508240579fb53b,7086394,B,0.127792,0.000000,genxii_v2,2024-03-18 18:46:07.590,1672,...,0.000000,0.000000,0.000000,N/A,2024-03-18 22:50:54.973,2024-03-24 22:56:15.210,GOPFS\gmsaprodsql$,GOPFS\gmsaprodsql$,3207,1
2,7089886,7,65f88c014800b0c9b782b375,7089886,C,0.140202,0.000000,genxii_v2,2024-03-18 18:46:25.657,8471,...,0.000000,0.000000,0.000000,N/A,2024-03-18 22:50:54.973,2024-03-24 22:56:15.210,GOPFS\gmsaprodsql$,GOPFS\gmsaprodsql$,3207,1
3,7090584,7,65f88c1843508240579fb550,7090584,B,0.128861,0.000000,genxii_v2,2024-03-18 18:46:48.347,2628,...,0.000000,0.000000,0.000000,N/A,2024-03-18 22:50:54.973,2024-03-24 22:56:15.210,GOPFS\gmsaprodsql$,GOPFS\gmsaprodsql$,3207,1
4,7090721,13,65f88c044800b0c9b782b376,7090721,B,0.073990,0.000000,genxii_v2,2024-03-18 18:46:28.840,6395,...,0.000000,0.000000,0.000000,N/A,2024-03-18 22:50:54.973,NaT,GOPFS\gmsaprodsql$,None,3201,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1482,7704145,7,66007dc344714f83adff0cef,7704145,A,0.056595,0.000000,genxii_v2,2024-03-24 19:23:47.010,1037,...,0.000000,0.000000,0.000000,N/A,2024-03-24 22:56:15.210,NaT,GOPFS\gmsaprodsql$,None,3207,1
1483,7704200,7,66008bc84800b0c9b7833565,7704200,D,0.172998,0.000000,genxii_v2,2024-03-24 20:23:36.873,894,...,0.000000,0.000000,0.000000,N/A,2024-03-24 22:56:15.210,NaT,GOPFS\gmsaprodsql$,None,3207,1
1484,7704233,7,6600944e44714f83adff0da6,7704233,A1,0.024187,0.000000,genxii_v2,2024-03-24 20:59:58.257,1161,...,0.000000,0.000000,0.000000,N/A,2024-03-24 22:56:15.210,NaT,GOPFS\gmsaprodsql$,None,3207,1
1485,7704277,7,6600c0484800b0c9b783360e,7704277,C,0.147384,0.000000,genxii_v2,2024-03-25 00:07:36.573,471,...,0.000000,0.000000,0.000000,N/A,2024-03-24 22:56:15.210,NaT,GOPFS\gmsaprodsql$,None,3207,1


### Subset

In [8]:
list_cols = [
    'bigaccountid',
    'intType',
]
df = df[list_cols].copy()
df

,bigaccountid,intType
0,7034142,13
1,7086394,13
2,7089886,7
3,7090584,7
4,7090721,13
...,...,...
1482,7704145,7
1483,7704200,7
1484,7704233,7
1485,7704277,7


### Save as parquet

In [9]:
%%time

# save
str_filename = 'df_scores.gzip'
str_local_path = f'{str_dirname_output}/{str_filename}'
df.to_parquet(str_local_path, compression='gzip')

Wall time: 100 ms


### Upload to s3

In [10]:
%%time

# upload
upload_to_s3(
    aws_access_key_id=AWS_ACCESS_KEY_ID, 
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY, 
    str_local_path=str_local_path, 
    str_bucket_key=f'{str_task}/{str_subtask}/{str_filename}', 
    str_bucket_name=str_project,
)

Wall time: 468 ms


### Clean-up

In [11]:
os.remove(str_local_path)